In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
files = [
    "optimal_FlexSIPP_2026-03-20_seed42",
    "optimal_@MAEDeR_2026-03-20_seed42",
]

result_file = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output", "table_comparison.tex")
all_results = {}
for file in files:
    algorithm = file.split("_")[1]
    filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output", f"{file}.json")
    result = json.load(open(filepath, "r"))
    for scenario, data in result.items():
        if "delay0" in data:
            if scenario in all_results:
                all_results[scenario][algorithm] = data["delay0"]
            else:
                all_results[scenario] = {algorithm: data["delay0"]}

In [ ]:
df = pd.DataFrame(columns=["Map", "#Agents", "Delay", "FlexSIPP", "@MAEDeR"])
for i, scen in enumerate(all_results):
    if "unique_routes_safe" in all_results[scen]["FlexSIPP"] and all_results[scen]["FlexSIPP"]["unique_routes_safe"]:
        total_delay_flexsipp = sum([all_results[scen]["FlexSIPP"]["final_paths"][a]["arrival"][1] - all_results[scen]["FlexSIPP"]["initial_paths"][a]["arrival"][1] for a in all_results[scen]["FlexSIPP"]["final_paths"]])
    else:
        total_delay_flexsipp = np.nan
    if "unique_routes_safe" in all_results[scen]["@MAEDeR"] and all_results[scen]["@MAEDeR"]["unique_routes_safe"]:
        total_delay_maeder = sum([all_results[scen]["@MAEDeR"]["final_paths"][a]["arrival"][1] - all_results[scen]["@MAEDeR"]["initial_paths"][a]["arrival"][1] for a in all_results[scen]["@MAEDeR"]["final_paths"]])
    else:
        total_delay_maeder = np.nan
    map_name = scen.split("-")[0] + scen.split("-")[-4]
    num_agents = scen.split("-")[-1].split("_")[0]
    df.loc[i] = [
        map_name,
        num_agents,
        all_results[scen]["FlexSIPP"]["delay"],
        total_delay_flexsipp,
        total_delay_maeder
    ]
with open(result_file, "w") as f:
    f.write(df.to_latex(index=False, float_format="%.2f", caption="Total delay for all agents combined.", label="tab:delays", position="t").replace("_paths", "").replace("-random-", "-r-").replace("-even-", "-e-"))
df